In [ ]:
# ============================================================
# STAGE 4 — POST-PROCESSING & VALIDATION
# D6 — Branch B — Structural Conversion
# ============================================================
#
# Validation basis:
# - Fixed Stage 1 document-grounded reference dataset
# - Branch B combined parsed extraction
# - Branch B technical diagnostics
# - D6 comparison rules frozen from Validation A
# ============================================================

from google.colab import files
from pathlib import Path
from difflib import SequenceMatcher

import hashlib
import json
import math
import re
import unicodedata

import pandas as pd
from scipy.optimize import linear_sum_assignment

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D6"
DOCUMENT_NAME = "Microsoft FY24 Q1 Press Release"

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

EXPECTED_REFERENCE_RECORD_COUNT = 147

EXPECTED_CATEGORY_COUNTS = {
    "Narrative performance highlight": 24,
    "Financial performance reconciliation": 4,
    "Segment revenue reconciliation": 3,
    "Selected product and service reconciliation": 15,
    "Income statement": 19,
    "Comprehensive income statement": 6,
    "Balance sheet": 34,
    "Cash flow statement": 34,
    "Segment revenue and operating income": 8
}

FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change",
    "Unit",
    "Reporting Period",
    "Source Location"
]

ALIGNMENT_BLOCK_FIELDS = [
    "Category",
    "Source Location"
]

MATCHING_IDENTITY_FIELDS = [
    "Metric",
    "Business Area",
    "Statement or Section",
    "Reporting Period"
]

ALIGNMENT_IDENTITY_FIELDS = (
    ALIGNMENT_BLOCK_FIELDS
    + MATCHING_IDENTITY_FIELDS
)

PRIMARY_CORRECTNESS_FIELDS = [
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change",
    "Unit"
]

NUMERIC_FIELDS = [
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change"
]

MATCH_SCORE_THRESHOLD = 0.34

OUTPUT_DIR = Path(
    "outputs_D6_validation_branch_B"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH, "-", BRANCH_NAME)
print(
    "Expected reference records:",
    EXPECTED_REFERENCE_RECORD_COUNT
)
print(
    "Alignment identity fields:",
    ALIGNMENT_IDENTITY_FIELDS
)
print(
    "Primary correctness fields:",
    PRIMARY_CORRECTNESS_FIELDS
)

In [ ]:
# ============================================================
# 2. Upload validation inputs
# ============================================================
# Required:
#   1) D6_reference_values.csv
#   2) D6_branch_B_combined_parsed_extraction.json
#   3) D6_branch_B_technical_diagnostics.json

uploaded = files.upload()

uploaded_files = list(
    uploaded.keys()
)

csv_files = [
    name
    for name in uploaded_files
    if name.lower().endswith(".csv")
]

json_files = [
    name
    for name in uploaded_files
    if name.lower().endswith(".json")
]

if len(csv_files) != 1:
    raise ValueError(
        "Upload exactly one D6 Stage 1 reference CSV."
    )

if len(json_files) != 2:
    raise ValueError(
        "Upload exactly two JSON files: the Branch B combined "
        "parsed extraction and technical diagnostics."
    )

REFERENCE_FILE = csv_files[0]

PARSED_EXTRACTION_FILE = None
TECHNICAL_DIAGNOSTICS_FILE = None


for file_name in json_files:

    with open(
        file_name,
        "r",
        encoding="utf-8-sig"
    ) as f:
        obj = json.load(f)

    if not isinstance(
        obj,
        dict
    ):
        continue

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and isinstance(
            obj.get("records"),
            list
        )
    ):
        PARSED_EXTRACTION_FILE = (
            file_name
        )

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "structurally_evaluable" in obj
        and "record_schema_valid" in obj
        and "valid_json" in obj
    ):
        TECHNICAL_DIAGNOSTICS_FILE = (
            file_name
        )


if PARSED_EXTRACTION_FILE is None:
    raise ValueError(
        "Could not identify the D6 Branch B "
        "combined parsed extraction."
    )

if TECHNICAL_DIAGNOSTICS_FILE is None:
    raise ValueError(
        "Could not identify the D6 Branch B "
        "technical diagnostics."
    )


print(
    "Reference:",
    REFERENCE_FILE
)

print(
    "Parsed extraction:",
    PARSED_EXTRACTION_FILE
)

print(
    "Technical diagnostics:",
    TECHNICAL_DIAGNOSTICS_FILE
)

In [ ]:
# ============================================================
# 3. Load inputs and verify identity/provenance
# ============================================================

reference_df = pd.read_csv(
    REFERENCE_FILE,
    dtype=object,
    keep_default_na=False,
    encoding="utf-8-sig"
)

def restore_csv_null(value):
    if value is None:
        return None
    if isinstance(value, str) and value == "":
        return None
    return value

def restore_mixed_number(value):
    if value is None or not isinstance(value, str):
        return value

    text = (
        value.strip()
        .replace(",", "")
        .replace("−", "-")
        .replace("–", "-")
    )

    if re.fullmatch(r"-?\d+", text):
        return int(text)

    if re.fullmatch(r"-?\d+\.\d+", text):
        return float(text)

    return value

for column in reference_df.columns:
    reference_df[column] = reference_df[column].map(restore_csv_null)

for field in NUMERIC_FIELDS:
    reference_df[field] = reference_df[field].map(restore_mixed_number)

with open(PARSED_EXTRACTION_FILE, "r", encoding="utf-8-sig") as f:
    extraction_json = json.load(f)

with open(TECHNICAL_DIAGNOSTICS_FILE, "r", encoding="utf-8-sig") as f:
    technical_diagnostics = json.load(f)

for artefact_name, artefact in {
    "parsed extraction": extraction_json,
    "technical diagnostics": technical_diagnostics
}.items():
    if artefact.get("document_id") != DOCUMENT_ID:
        raise ValueError(f"Unexpected {artefact_name} document_id.")
    if artefact.get("branch") != BRANCH:
        raise ValueError(f"Unexpected {artefact_name} branch.")

extracted_df = pd.DataFrame(extraction_json["records"])

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_provenance = {
    "reference_file": REFERENCE_FILE,
    "reference_sha256": sha256_file(REFERENCE_FILE),
    "parsed_extraction_file": PARSED_EXTRACTION_FILE,
    "parsed_extraction_sha256": sha256_file(PARSED_EXTRACTION_FILE),
    "technical_diagnostics_file": TECHNICAL_DIAGNOSTICS_FILE,
    "technical_diagnostics_sha256": sha256_file(TECHNICAL_DIAGNOSTICS_FILE)
}

print("Reference shape:", reference_df.shape)
print("Extraction shape:", extracted_df.shape)


In [ ]:
# ============================================================
# 4. Import Branch B technical/schema status
# ============================================================

structurally_evaluable = bool(
    technical_diagnostics.get("structurally_evaluable", False)
)

schema_validity = bool(structurally_evaluable)

schema_diagnostics = {
    "valid_json": bool(technical_diagnostics.get("valid_json", False)),
    "record_schema_valid": bool(
        technical_diagnostics.get("record_schema_valid", False)
    ),
    "field_types_valid": bool(
        technical_diagnostics.get("field_types_valid", False)
    ),
    "structurally_evaluable": structurally_evaluable,
    "schema_validity": schema_validity
}

if not structurally_evaluable:
    raise ValueError(
        "D6 Branch B output is not structurally evaluable. "
        "Content-level validation cannot proceed."
    )

print(json.dumps(schema_diagnostics, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 5. Verify fixed reference and prepare extraction copy
# ============================================================

if reference_df.columns.tolist() != FIELDS:
    raise ValueError(
        "D6 Stage 1 reference schema does not match the fixed field list."
    )

if len(reference_df) != EXPECTED_REFERENCE_RECORD_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_REFERENCE_RECORD_COUNT} reference records; "
        f"found {len(reference_df)}."
    )

reference_category_counts = (
    reference_df["Category"].value_counts().to_dict()
)

if reference_category_counts != EXPECTED_CATEGORY_COUNTS:
    raise ValueError(
        "D6 Stage 1 reference category counts do not match the fixed design."
    )

missing_extraction_fields = [
    field for field in FIELDS
    if field not in extracted_df.columns
]

extracted_comparison_df = extracted_df.copy(deep=True)

for field in missing_extraction_fields:
    extracted_comparison_df[field] = None

extracted_comparison_df = extracted_comparison_df[FIELDS].copy()
reference_comparison_df = reference_df[FIELDS].copy(deep=True)

def is_null(value):
    if value is None:
        return True
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False

extraction_record_count_valid = (
    len(extracted_comparison_df) == EXPECTED_REFERENCE_RECORD_COUNT
)

extraction_category_counts = (
    extracted_comparison_df["Category"]
    .value_counts(dropna=False)
    .to_dict()
)

extraction_category_counts_valid = (
    extraction_category_counts == EXPECTED_CATEGORY_COUNTS
)

content_diagnostics = {
    "reference_record_count_valid": True,
    "reference_category_counts_valid": True,
    "extraction_record_count_valid": bool(extraction_record_count_valid),
    "extraction_category_counts_valid": bool(extraction_category_counts_valid),
    "branch_B_scope_complete": technical_diagnostics.get("scope_complete"),
    "branch_B_content_diagnostics": technical_diagnostics.get(
        "content_diagnostics"
    )
}

print("Reference records:", len(reference_comparison_df))
print("Extracted records:", len(extracted_comparison_df))
print("Missing extraction columns:", missing_extraction_fields)
print(
    "Extraction record count matches reference:",
    extraction_record_count_valid
)


In [ ]:
# ============================================================
# 6. Controlled comparison normalisation
# ============================================================

def normalise_text(value):
    if is_null(value):
        return None

    text = unicodedata.normalize("NFKC", str(value))

    replacements = {
        "\u00a0": " ",
        "\u2007": " ",
        "\u202f": " ",
        "\u2010": "-",
        "\u2011": "-",
        "\u2012": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u2212": "-",
        "\u2018": "'",
        "\u2019": "'"
    }

    for source, target in replacements.items():
        text = text.replace(source, target)

    text = re.sub(r"\s+", " ", text)

    return text.strip().casefold()


STOPWORDS = {
    "a", "an", "and", "as", "at", "by", "for", "from", "in",
    "is", "of", "on", "or", "the", "to", "was", "were", "with"
}


def comparison_tokens(value):
    text = normalise_text(value)

    if text is None:
        return set()

    text = re.sub(r"[^a-z0-9]+", " ", text)

    return {
        token
        for token in text.split()
        if token and token not in STOPWORDS
    }


def sequence_similarity(first, second):
    first_text = normalise_text(first)
    second_text = normalise_text(second)

    if first_text is None and second_text is None:
        return 1.0

    if first_text is None or second_text is None:
        return 0.0

    return SequenceMatcher(
        None,
        first_text,
        second_text
    ).ratio()


def jaccard_similarity(first, second):
    first_tokens = comparison_tokens(first)
    second_tokens = comparison_tokens(second)

    if not first_tokens and not second_tokens:
        return 1.0

    if not first_tokens or not second_tokens:
        return 0.0

    return (
        len(first_tokens & second_tokens)
        / len(first_tokens | second_tokens)
    )


def text_similarity(first, second):

    return max(
        sequence_similarity(first, second),
        jaccard_similarity(first, second)
    )


In [ ]:
# ============================================================
# 7. Controlled D6 equivalence rules
# ============================================================

UNIT_EQUIVALENCE_MAP = {
    "usd billion": "usd billion",
    "usd billions": "usd billion",
    "billion usd": "usd billion",

    "usd millions": "usd millions",
    "usd millions; percent": "usd millions",

    "usd per share": "usd per share",
    "usd per share; percent": "usd per share",

    "percent": "percent",
    "percentage": "percent",

    "million subscribers": "million subscribers",

    "million shares": "million shares",
    "millions of shares": "million shares"
}


BUSINESS_AREA_EQUIVALENCE_MAP = {
    "total": "total",
    "corporate": "corporate",
    "microsoft": "corporate"
}


STATEMENT_EQUIVALENCE_MAP = {
    "quarterly results":
        "quarterly results and business highlights",

    "business highlights":
        "quarterly results and business highlights",

    "quarterly results and business highlights":
        "quarterly results and business highlights",

    "shareholder returns":
        "quarterly results and business highlights"
}


PERIOD_EQUIVALENCE_MAP = {
    "first quarter fiscal year 2024":
        "quarter ended september 30, 2023",

    "first quarter of fiscal year 2024":
        "quarter ended september 30, 2023",

    "quarter ended september 30, 2023":
        "quarter ended september 30, 2023",

    "three months ended september 30":
        "three months ended september 30, 2023",

    "three months ended september 30, 2023":
        "three months ended september 30, 2023",

    "september 30, 2023 and june 30, 2023":
        "september 30, 2023 and june 30, 2023"
}


def canonical_from_map(value, mapping):
    text = normalise_text(value)

    if text is None:
        return None

    return mapping.get(text, text)


def canonical_unit(value):
    return canonical_from_map(
        value,
        UNIT_EQUIVALENCE_MAP
    )


def canonical_business_area(value):
    return canonical_from_map(
        value,
        BUSINESS_AREA_EQUIVALENCE_MAP
    )


def canonical_statement(value):
    return canonical_from_map(
        value,
        STATEMENT_EQUIVALENCE_MAP
    )


def canonical_period(value):
    return canonical_from_map(
        value,
        PERIOD_EQUIVALENCE_MAP
    )


def canonical_metric(value):
    text = normalise_text(value)

    if text is None:
        return None

    text = text.replace(
        "weighted average shares outstanding - basic",
        "basic weighted average shares outstanding"
    )

    text = text.replace(
        "weighted average shares outstanding - diluted",
        "diluted weighted average shares outstanding"
    )

    return text


In [ ]:
# ============================================================
# 8. Controlled Metric + Business Area equivalence
# ============================================================

METRIC_BUSINESS_EQUIVALENCE_RAW = [

    # Microsoft Cloud
    (
        ("Microsoft Cloud revenue", "Microsoft Cloud"),
        ("Revenue", "Microsoft Cloud")
    ),

    # Office Commercial
    (
        (
            "Office Commercial products and cloud services revenue",
            "Office Commercial"
        ),
        (
            "Revenue",
            "Office Commercial products and cloud services"
        )
    ),

    # Office 365 Commercial
    (
        (
            "Office 365 Commercial revenue",
            "Office 365 Commercial"
        ),
        (
            "Revenue growth",
            "Office 365 Commercial"
        )
    ),

    # Office Consumer
    (
        (
            "Office Consumer products and cloud services revenue",
            "Office Consumer"
        ),
        (
            "Revenue",
            "Office Consumer products and cloud services"
        )
    ),

    # LinkedIn
    (
        ("LinkedIn revenue", "LinkedIn"),
        ("Revenue", "LinkedIn")
    ),

    # Dynamics products and cloud services
    (
        (
            "Dynamics products and cloud services revenue",
            "Dynamics"
        ),
        (
            "Revenue",
            "Dynamics products and cloud services"
        )
    ),

    # Dynamics 365
    (
        ("Dynamics 365 revenue", "Dynamics 365"),
        ("Revenue growth", "Dynamics 365")
    ),

    # Server products and cloud services
    (
        (
            "Server products and cloud services revenue",
            "Server products and cloud services"
        ),
        (
            "Revenue",
            "Server products and cloud services"
        )
    ),

    # Azure
    (
        (
            "Azure and other cloud services revenue",
            "Azure and other cloud services"
        ),
        (
            "Revenue growth",
            "Azure and other cloud services"
        )
    ),

    # Windows
    (
        ("Windows revenue", "Windows"),
        ("Revenue", "Windows")
    ),

    # Windows OEM
    (
        ("Windows OEM revenue", "Windows OEM"),
        ("Revenue growth", "Windows OEM")
    ),

    # Windows Commercial
    (
        (
            "Windows Commercial products and cloud services revenue",
            "Windows Commercial"
        ),
        (
            "Revenue growth",
            "Windows Commercial products and cloud services"
        )
    ),

    # Devices
    (
        ("Devices revenue", "Devices"),
        ("Revenue", "Devices")
    ),

    # Xbox
    (
        (
            "Xbox content and services revenue",
            "Xbox content and services"
        ),
        (
            "Revenue",
            "Xbox content and services"
        )
    ),

    # Search and news advertising — narrative
    (
        (
            "Search and news advertising revenue excluding traffic acquisition costs",
            "Search and news advertising"
        ),
        (
            "Revenue excluding traffic acquisition costs",
            "Search and news advertising"
        )
    ),

    # Search and news advertising — reconciliation table
    (
        (
            "Revenue",
            "Search and news advertising excluding traffic acquisition costs"
        ),
        (
            "Revenue excluding traffic acquisition costs",
            "Search and news advertising"
        )
    )
]


def normalise_metric_business_pair(metric, business_area):
    return (
        canonical_metric(metric),
        canonical_business_area(business_area)
    )


METRIC_BUSINESS_EQUIVALENCE = {
    (
        normalise_metric_business_pair(
            reference_metric,
            reference_business
        ),
        normalise_metric_business_pair(
            extracted_metric,
            extracted_business
        )
    )
    for (
        (reference_metric, reference_business),
        (extracted_metric, extracted_business)
    )
    in METRIC_BUSINESS_EQUIVALENCE_RAW
}


def is_metric_business_pair_equivalent(
    reference_metric,
    reference_business,
    extracted_metric,
    extracted_business
):
    reference_pair = normalise_metric_business_pair(
        reference_metric,
        reference_business
    )

    extracted_pair = normalise_metric_business_pair(
        extracted_metric,
        extracted_business
    )

    return (
        reference_pair,
        extracted_pair
    ) in METRIC_BUSINESS_EQUIVALENCE


In [ ]:
# ============================================================
# 9. Numeric and exact-value comparison
# ============================================================

def numeric_value(value):
    if is_null(value) or isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    if isinstance(value, str):
        text = (
            value.strip()
            .replace(",", "")
            .replace("−", "-")
            .replace("–", "-")
        )

        if re.fullmatch(r"-?\d+(?:\.\d+)?", text):
            return float(text)

    return None


def values_match(reference_value, extracted_value, tolerance=1e-9):

    # Both absent = agreement
    if is_null(reference_value) and is_null(extracted_value):
        return True

    # Only one absent = discrepancy
    if is_null(reference_value) or is_null(extracted_value):
        return False

    reference_number = numeric_value(reference_value)
    extracted_number = numeric_value(extracted_value)

    if reference_number is not None and extracted_number is not None:
        return math.isclose(
            reference_number,
            extracted_number,
            rel_tol=tolerance,
            abs_tol=tolerance
        )

    return (
        normalise_text(reference_value)
        == normalise_text(extracted_value)
    )


In [ ]:
# ============================================================
# 10. Prepare comparison blocks
# ============================================================

reference_comparison_df["_reference_index"] = range(
    len(reference_comparison_df)
)

extracted_comparison_df["_extraction_index"] = range(
    len(extracted_comparison_df)
)

for dataframe in [
    reference_comparison_df,
    extracted_comparison_df
]:
    dataframe["_block_category"] = (
        dataframe["Category"].map(normalise_text)
    )

    dataframe["_block_source"] = (
        dataframe["Source Location"].map(normalise_text)
    )

    dataframe["_matching_block"] = list(
        zip(
            dataframe["_block_category"],
            dataframe["_block_source"]
        )
    )

print("Comparison blocks prepared.")
print("Blocking fields:", ALIGNMENT_BLOCK_FIELDS)


In [ ]:
# ============================================================
# 11. Deterministic identity-based matching score
# ============================================================


def matching_score(reference_row, extracted_row):

    metric_score = text_similarity(
        canonical_metric(reference_row["Metric"]),
        canonical_metric(extracted_row["Metric"])
    )

    business_score = (
        1.0
        if canonical_business_area(reference_row["Business Area"])
        == canonical_business_area(extracted_row["Business Area"])
        else text_similarity(
            reference_row["Business Area"],
            extracted_row["Business Area"]
        )
    )

    statement_score = (
        1.0
        if canonical_statement(reference_row["Statement or Section"])
        == canonical_statement(extracted_row["Statement or Section"])
        else text_similarity(
            reference_row["Statement or Section"],
            extracted_row["Statement or Section"]
        )
    )

    period_score = (
        1.0
        if canonical_period(reference_row["Reporting Period"])
        == canonical_period(extracted_row["Reporting Period"])
        else text_similarity(
            reference_row["Reporting Period"],
            extracted_row["Reporting Period"]
        )
    )


    total_score = (
        0.55 * metric_score
        + 0.25 * business_score
        + 0.15 * statement_score
        + 0.05 * period_score
    )

    return {
        "total": total_score,
        "metric": metric_score,
        "business_area": business_score,
        "statement": statement_score,
        "reporting_period": period_score
    }


In [ ]:
# ============================================================
# 12. One-to-one record alignment
# ============================================================

all_blocks = sorted(
    set(reference_comparison_df["_matching_block"])
    | set(extracted_comparison_df["_matching_block"]),
    key=str
)

matched_pairs = []

unmatched_reference_indices = set(
    reference_comparison_df["_reference_index"].tolist()
)

unmatched_extraction_indices = set(
    extracted_comparison_df["_extraction_index"].tolist()
)


for block in all_blocks:

    reference_block = reference_comparison_df.loc[
        reference_comparison_df["_matching_block"] == block
    ]

    extraction_block = extracted_comparison_df.loc[
        extracted_comparison_df["_matching_block"] == block
    ]

    if reference_block.empty or extraction_block.empty:
        continue

    reference_rows = [
        row
        for _, row in reference_block.iterrows()
    ]

    extraction_rows = [
        row
        for _, row in extraction_block.iterrows()
    ]

    score_matrix = []
    details_matrix = []

    for reference_row in reference_rows:

        score_row = []
        details_row = []

        for extracted_row in extraction_rows:
            details = matching_score(
                reference_row,
                extracted_row
            )

            score_row.append(details["total"])
            details_row.append(details)

        score_matrix.append(score_row)
        details_matrix.append(details_row)

    cost_matrix = [
        [1.0 - score for score in row]
        for row in score_matrix
    ]

    row_positions, column_positions = linear_sum_assignment(
        cost_matrix
    )

    for row_position, column_position in zip(
        row_positions,
        column_positions
    ):

        details = details_matrix[
            row_position
        ][
            column_position
        ]

        if details["total"] < MATCH_SCORE_THRESHOLD:
            continue

        reference_index = int(
            reference_rows[
                row_position
            ][
                "_reference_index"
            ]
        )

        extraction_index = int(
            extraction_rows[
                column_position
            ][
                "_extraction_index"
            ]
        )

        matched_pairs.append(
            {
                "reference_index": reference_index,
                "extraction_index": extraction_index,
                "matching_score": details["total"],
                "metric_matching_score": details["metric"],
                "business_area_matching_score":
                    details["business_area"],
                "statement_matching_score": details["statement"],
                "reporting_period_matching_score":
                    details["reporting_period"]
            }
        )

        unmatched_reference_indices.discard(
            reference_index
        )

        unmatched_extraction_indices.discard(
            extraction_index
        )


aligned_record_count = len(matched_pairs)
missing_record_count = len(unmatched_reference_indices)
unsupported_record_count = len(unmatched_extraction_indices)

print("Aligned records:", aligned_record_count)
print("Missing records:", missing_record_count)
print(
    "Unsupported/unmatched extracted records:",
    unsupported_record_count
)


In [ ]:
# ============================================================
# 13. Missing and unsupported/unmatched records
# ============================================================

missing_records_df = (
    reference_comparison_df.loc[
        reference_comparison_df["_reference_index"].isin(
            unmatched_reference_indices
        ),
        ["_reference_index"] + FIELDS
    ].copy()
)

unsupported_records_df = (
    extracted_comparison_df.loc[
        extracted_comparison_df["_extraction_index"].isin(
            unmatched_extraction_indices
        ),
        ["_extraction_index"] + FIELDS
    ].copy()
)

print("Missing:", len(missing_records_df))
print("Unsupported/unmatched:", len(unsupported_records_df))


In [ ]:
# ============================================================
# 14. Field-level correctness after alignment
# ============================================================

def exact_text_match(first, second):
    return normalise_text(first) == normalise_text(second)


def mapped_text_match(first, second, canonical_function):
    return canonical_function(first) == canonical_function(second)


comparison_rows = []

for pair in matched_pairs:

    reference_row = reference_comparison_df.loc[
        reference_comparison_df["_reference_index"]
        == pair["reference_index"]
    ].iloc[0]

    extracted_row = extracted_comparison_df.loc[
        extracted_comparison_df["_extraction_index"]
        == pair["extraction_index"]
    ].iloc[0]

    category_match = exact_text_match(
        reference_row["Category"],
        extracted_row["Category"]
    )

    statement_match = mapped_text_match(
        reference_row["Statement or Section"],
        extracted_row["Statement or Section"],
        canonical_statement
    )

    metric_similarity = text_similarity(
        canonical_metric(reference_row["Metric"]),
        canonical_metric(extracted_row["Metric"])
    )

    metric_business_pair_rule_applied = (
        is_metric_business_pair_equivalent(
            reference_row["Metric"],
            reference_row["Business Area"],
            extracted_row["Metric"],
            extracted_row["Business Area"]
        )
    )


    metric_match = (
        canonical_metric(reference_row["Metric"])
        == canonical_metric(extracted_row["Metric"])
        or metric_business_pair_rule_applied
    )


    business_area_match = (
        canonical_business_area(
            reference_row["Business Area"]
        )
        == canonical_business_area(
            extracted_row["Business Area"]
        )
        or metric_business_pair_rule_applied
    )

    value_2023_match = values_match(
        reference_row["Value 2023"],
        extracted_row["Value 2023"]
    )

    value_2022_match = values_match(
        reference_row["Value 2022"],
        extracted_row["Value 2022"]
    )

    gaap_change_match = values_match(
        reference_row["GAAP YoY Change"],
        extracted_row["GAAP YoY Change"]
    )

    constant_currency_impact_match = values_match(
        reference_row["Constant Currency Impact"],
        extracted_row["Constant Currency Impact"]
    )

    constant_currency_change_match = values_match(
        reference_row["Constant Currency YoY Change"],
        extracted_row["Constant Currency YoY Change"]
    )

    unit_match = mapped_text_match(
        reference_row["Unit"],
        extracted_row["Unit"],
        canonical_unit
    )

    reporting_period_match = mapped_text_match(
        reference_row["Reporting Period"],
        extracted_row["Reporting Period"],
        canonical_period
    )

    source_location_match = exact_text_match(
        reference_row["Source Location"],
        extracted_row["Source Location"]
    )

    field_matches = {
        "Category": category_match,
        "Statement or Section": statement_match,
        "Metric": metric_match,
        "Business Area": business_area_match,
        "Value 2023": value_2023_match,
        "Value 2022": value_2022_match,
        "GAAP YoY Change": gaap_change_match,
        "Constant Currency Impact":
            constant_currency_impact_match,
        "Constant Currency YoY Change":
            constant_currency_change_match,
        "Unit": unit_match,
        "Reporting Period": reporting_period_match,
        "Source Location": source_location_match
    }

    all_primary_fields_match = all(
        field_matches[field]
        for field in PRIMARY_CORRECTNESS_FIELDS
    )

    all_mismatched_fields = [
        field
        for field, match in field_matches.items()
        if not match
    ]

    primary_mismatched_fields = [
        field
        for field in PRIMARY_CORRECTNESS_FIELDS
        if not field_matches[field]
    ]

    output_row = {
        "Reference Index": pair["reference_index"],
        "Extraction Index": pair["extraction_index"],
        "Matching Score": pair["matching_score"],

        "Metric Matching Score":
            pair["metric_matching_score"],

        "Business Area Matching Score":
            pair["business_area_matching_score"],

        "Statement Matching Score":
            pair["statement_matching_score"],

        "Reporting Period Matching Score":
            pair["reporting_period_matching_score"],

        "Category":
            reference_row["Category"],

        "Reference Metric":
            reference_row["Metric"],

        "Extracted Metric":
            extracted_row["Metric"],

        "Metric Lexical Similarity":
            metric_similarity,

        "Metric-Business Pair Rule Applied":
            bool(metric_business_pair_rule_applied),

        "Fully Correct":
            bool(all_primary_fields_match),

        "all_mismatched_fields":
            ", ".join(all_mismatched_fields),

        "primary_mismatched_fields":
            ", ".join(primary_mismatched_fields)
    }



    for field in FIELDS:
        output_row[f"Reference {field}"] = reference_row[field]
        output_row[f"Extracted {field}"] = extracted_row[field]
        output_row[f"{field} Match"] = bool(field_matches[field])

    comparison_rows.append(output_row)


comparison_df = pd.DataFrame(comparison_rows)

print("Compared aligned records:", len(comparison_df))
print(
    "Fully correct:",
    int(comparison_df["Fully Correct"].sum())
)

display(comparison_df.head(10))


In [ ]:
# ============================================================
# 15. Fully correct and discrepant aligned records
# ============================================================

fully_correct_records_df = comparison_df.loc[
    comparison_df["Fully Correct"] == True
].copy()

discrepant_records_df = comparison_df.loc[
    comparison_df["Fully Correct"] == False
].copy()

print("Fully correct aligned records:", len(fully_correct_records_df))
print("Discrepant aligned records:", len(discrepant_records_df))

if not discrepant_records_df.empty:
    display(
        discrepant_records_df[
            [
                "Reference Index",
                "Extraction Index",
                "Category",
                "Reference Metric",
                "Extracted Metric",
                "primary_mismatched_fields"
            ]
        ].head(30)
    )


In [ ]:
# ============================================================
# 16. Field-level discrepancy diagnostics
# ============================================================

field_discrepancy_rows = []

for _, row in comparison_df.iterrows():

    for field in FIELDS:

        if not bool(row[f"{field} Match"]):
            field_discrepancy_rows.append(
                {
                    "Reference Index":
                        int(row["Reference Index"]),
                    "Extraction Index":
                        int(row["Extraction Index"]),
                    "Category":
                        row["Category"],
                    "Reference Metric":
                        row["Reference Metric"],
                    "Extracted Metric":
                        row["Extracted Metric"],
                    "Field":
                        field,
                    "Reference Value":
                        row[f"Reference {field}"],
                    "Extracted Value":
                        row[f"Extracted {field}"]
                }
            )


field_discrepancies_df = pd.DataFrame(
    field_discrepancy_rows
)

print(
    "Field discrepancy count:",
    len(field_discrepancies_df)
)

if not field_discrepancies_df.empty:
    display(field_discrepancies_df)


In [ ]:
# ============================================================
# 17. Field-level accuracy diagnostics
# ============================================================

field_accuracy_rows = []

for field in FIELDS:
    correct_count = int(
        comparison_df[f"{field} Match"].sum()
    )

    aligned_count = len(comparison_df)

    field_accuracy_rows.append({
        "Field": field,
        "used_in_alignment_block": field in ALIGNMENT_BLOCK_FIELDS,
        "used_in_matching_identity": field in MATCHING_IDENTITY_FIELDS,
        "used_in_primary_correctness": field in PRIMARY_CORRECTNESS_FIELDS,
        "Correct Records": correct_count,
        "Aligned Records": aligned_count,
        "Accuracy": (
            correct_count / aligned_count
            if aligned_count > 0
            else None
        ),
        "Overall Accuracy Against Reference": (
            correct_count / len(reference_comparison_df)
            if len(reference_comparison_df) > 0
            else None
        )
    })

field_accuracy_df = pd.DataFrame(field_accuracy_rows)

field_accuracy_dictionary = {
    row["Field"]: (
        float(row["Accuracy"])
        if pd.notna(row["Accuracy"])
        else None
    )
    for _, row in field_accuracy_df.iterrows()
}

display(field_accuracy_df)


In [ ]:
# ============================================================
# 18. Calculate common validation metrics
# ============================================================

fully_correct_record_count = int(
    comparison_df["Fully Correct"].sum()
)

discrepant_record_count = (
    aligned_record_count - fully_correct_record_count
)

N_REF = int(len(reference_comparison_df))
N_EXT = int(len(extracted_comparison_df))

completeness = (
    aligned_record_count / N_REF
    if N_REF else 0.0
)

missing_rate = (
    missing_record_count / N_REF
    if N_REF else 0.0
)

record_precision_exact = (
    fully_correct_record_count / N_EXT
    if N_EXT else 0.0
)

record_recall_exact = (
    fully_correct_record_count / N_REF
    if N_REF else 0.0
)

record_f1_exact = (
    2 * record_precision_exact * record_recall_exact
    / (record_precision_exact + record_recall_exact)
    if (record_precision_exact + record_recall_exact)
    else 0.0
)

unsupported_rate = (
    unsupported_record_count / N_EXT
    if N_EXT else 0.0
)

discrepancy_rate_among_aligned = (
    discrepant_record_count / aligned_record_count
    if aligned_record_count else 0.0
)

correct_field_instances = int(
    sum(
        comparison_df[f"{field} Match"].sum()
        for field in PRIMARY_CORRECTNESS_FIELDS
    )
)

expected_field_instances = int(
    N_REF * len(PRIMARY_CORRECTNESS_FIELDS)
)

field_accuracy = (
    correct_field_instances / expected_field_instances
    if expected_field_instances else 0.0
)

print("Fully correct records:", fully_correct_record_count)
print("Discrepant records:", discrepant_record_count)
print("Completeness:", completeness)
print("Exact record precision:", record_precision_exact)
print("Exact record recall:", record_recall_exact)
print("Exact record F1:", record_f1_exact)
print("Field accuracy:", field_accuracy)


In [ ]:
# ============================================================
# 19. Category-level metrics
# ============================================================

category_metric_rows = []

for category, expected_count in EXPECTED_CATEGORY_COUNTS.items():

    category_comparison = comparison_df.loc[
        comparison_df["Category"] == category
    ]

    extracted_category_count = int(
        (
            extracted_df["Category"]
            == category
        ).sum()
    )

    aligned_count = len(category_comparison)

    fully_correct_count = int(
        category_comparison[
            "Fully Correct"
        ].sum()
    )

    discrepant_count = (
        aligned_count - fully_correct_count
    )

    category_completeness = (
        aligned_count / expected_count
        if expected_count > 0
        else None
    )

    category_precision_exact = (
        fully_correct_count
        / extracted_category_count
        if extracted_category_count > 0
        else 0.0
    )

    category_recall_exact = (
        fully_correct_count
        / expected_count
        if expected_count > 0
        else 0.0
    )

    category_f1_exact = (
        2
        * category_precision_exact
        * category_recall_exact
        / (
            category_precision_exact
            + category_recall_exact
        )
        if (
            category_precision_exact
            + category_recall_exact
            > 0
        )
        else 0.0
    )

    category_metric_rows.append(
        {
            "Category": category,
            "Expected Records": expected_count,
            "Extracted Records": extracted_category_count,
            "Aligned Records": aligned_count,
            "Fully Correct Records": fully_correct_count,
            "Discrepant Records": discrepant_count,
            "Completeness": category_completeness,
            "Record Precision Exact":
                category_precision_exact,
            "Record Recall Exact":
                category_recall_exact,
            "Record F1 Exact":
                category_f1_exact
        }
    )


category_metrics_df = pd.DataFrame(
    category_metric_rows
)

display(category_metrics_df)


In [ ]:
# ============================================================
# 20. Build final Branch B validation summary
# ============================================================

category_metrics_dictionary = {
    row["Category"]: {
        "expected_records": int(row["Expected Records"]),
        "extracted_records": int(row["Extracted Records"]),
        "aligned_records": int(row["Aligned Records"]),
        "fully_correct_records": int(row["Fully Correct Records"]),
        "discrepant_records": int(row["Discrepant Records"]),
        "completeness": float(row["Completeness"]),
        "record_precision_exact": float(row["Record Precision Exact"]),
        "record_recall_exact": float(row["Record Recall Exact"]),
        "record_f1_exact": float(row["Record F1 Exact"])
    }
    for _, row in category_metrics_df.iterrows()
}

summary = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "reference_records": N_REF,
    "extracted_records": N_EXT,
    "aligned_records": int(aligned_record_count),
    "fully_correct_records": fully_correct_record_count,
    "discrepant_records": discrepant_record_count,
    "missing_records": int(missing_record_count),
    "unsupported_extracted_records": int(unsupported_record_count),
    "completeness": round(completeness, 4),
    "missing_rate": round(missing_rate, 4),
    "record_precision_exact": round(record_precision_exact, 4),
    "record_recall_exact": round(record_recall_exact, 4),
    "record_f1_exact": round(record_f1_exact, 4),
    "unsupported_rate": round(unsupported_rate, 4),
    "discrepancy_rate_among_aligned": round(
        discrepancy_rate_among_aligned, 4
    ),
    "field_accuracy": round(field_accuracy, 4),
    "field_accuracy_among_aligned": {
        field: (
            round(value, 4)
            if value is not None
            else None
        )
        for field, value in field_accuracy_dictionary.items()
    },
    "schema_validity": schema_validity,
    "schema_diagnostics": schema_diagnostics,
    "structurally_evaluable": structurally_evaluable,
    "content_diagnostics": content_diagnostics,
    "alignment_block_fields": ALIGNMENT_BLOCK_FIELDS,
    "matching_identity_fields": MATCHING_IDENTITY_FIELDS,
    "alignment_identity_fields": ALIGNMENT_IDENTITY_FIELDS,
    "primary_correctness_fields": PRIMARY_CORRECTNESS_FIELDS,
    "matching_rules": {
        "blocking_fields": ALIGNMENT_BLOCK_FIELDS,
        "identity_fields": MATCHING_IDENTITY_FIELDS,
        "matching_method": (
            "One-to-one Hungarian assignment within "
            "Category + Source Location blocks"
        ),
        "minimum_score": MATCH_SCORE_THRESHOLD,
        "value_used_for_alignment": False,
        "unit_used_for_alignment": False,
        "gaap_change_used_for_alignment": False,
        "constant_currency_outcomes_used_for_alignment": False
    },
    "comparison_rules_frozen_from_branch_A": True,
    "normalisation_note": (
        "Controlled deterministic normalisation and source-grounded "
        "equivalence rules are applied only to comparison copies. "
        "The preserved Branch B extraction is not modified."
    ),
    "category_metrics": category_metrics_dictionary,
    "input_provenance": input_provenance
}

print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 21. Validation integrity checks
# ============================================================

assert aligned_record_count + missing_record_count == N_REF
assert aligned_record_count + unsupported_record_count == N_EXT
assert (
    fully_correct_record_count + discrepant_record_count
    == aligned_record_count
)

for metric_name, metric_value in {
    "completeness": completeness,
    "missing_rate": missing_rate,
    "record_precision_exact": record_precision_exact,
    "record_recall_exact": record_recall_exact,
    "record_f1_exact": record_f1_exact,
    "unsupported_rate": unsupported_rate,
    "discrepancy_rate": discrepancy_rate_among_aligned,
    "field_accuracy": field_accuracy
}.items():
    assert 0.0 <= metric_value <= 1.0, (
        f"Invalid {metric_name}: {metric_value}"
    )

print("Validation integrity checks passed.")


In [ ]:
# ============================================================
# 22. Export validation artefacts
# ============================================================

DETAILED_PATH = (
    OUTPUT_DIR / "D6_branch_B_validation_detailed.csv"
)

FULLY_CORRECT_PATH = (
    OUTPUT_DIR / "D6_branch_B_fully_correct_records.csv"
)

DISCREPANT_PATH = (
    OUTPUT_DIR / "D6_branch_B_discrepant_records.csv"
)

MISSING_PATH = (
    OUTPUT_DIR / "D6_branch_B_missing_records.csv"
)

UNSUPPORTED_PATH = (
    OUTPUT_DIR / "D6_branch_B_unsupported_records.csv"
)

FIELD_DISCREPANCIES_PATH = (
    OUTPUT_DIR / "D6_branch_B_field_discrepancies.csv"
)

FIELD_ACCURACY_PATH = (
    OUTPUT_DIR / "D6_branch_B_field_accuracy.csv"
)

CATEGORY_METRICS_PATH = (
    OUTPUT_DIR / "D6_branch_B_category_metrics.csv"
)

SUMMARY_PATH = (
    OUTPUT_DIR / "D6_branch_B_validation_summary.json"
)

comparison_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig"
)

fully_correct_records_df.to_csv(
    FULLY_CORRECT_PATH,
    index=False,
    encoding="utf-8-sig"
)

discrepant_records_df.to_csv(
    DISCREPANT_PATH,
    index=False,
    encoding="utf-8-sig"
)

missing_records_df.to_csv(
    MISSING_PATH,
    index=False,
    encoding="utf-8-sig"
)

unsupported_records_df.to_csv(
    UNSUPPORTED_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_discrepancies_df.to_csv(
    FIELD_DISCREPANCIES_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_accuracy_df.to_csv(
    FIELD_ACCURACY_PATH,
    index=False,
    encoding="utf-8-sig"
)

category_metrics_df.to_csv(
    CATEGORY_METRICS_PATH,
    index=False,
    encoding="utf-8-sig"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print("D6 Validation B artefacts saved.")


In [ ]:
# ============================================================
# 23. Download generated validation artefacts
# ============================================================

GENERATED_OUTPUTS = [
    DETAILED_PATH,
    FULLY_CORRECT_PATH,
    DISCREPANT_PATH,
    MISSING_PATH,
    UNSUPPORTED_PATH,
    FIELD_DISCREPANCIES_PATH,
    FIELD_ACCURACY_PATH,
    CATEGORY_METRICS_PATH,
    SUMMARY_PATH
]

for output_path in GENERATED_OUTPUTS:
    print(
        "-",
        output_path.name,
        "| exists:",
        output_path.exists()
    )

for output_path in GENERATED_OUTPUTS:
    if output_path.exists():
        files.download(output_path)
